# Notebook 3 — Grammar-Constrained Community Labeling and Stage 3 Intelligence

This version is built to satisfy the **mandatory grammar-constrained JSON** requirement.

## What is different from the previous version
- `USE_GBNF_GRAMMAR = True` by default
- lighter, more stable JSON schema
- shorter prompt
- smaller token budget per call
- one explicit grammar test before batch execution
- same batching, checkpointing, and multi-call community logic

## Why this version is safer
Some Colab runtimes become unstable when grammar constraints are combined with:
- very long prompts
- overly large JSON schemas
- long generations

This notebook reduces those risks while **still using grammar-constrained output**.


In [1]:
# ============================================================
# 1. INSTALL / MODEL SETUP
# ============================================================
!pip uninstall -y llama_cpp_python llama-cpp-python 2>/dev/null || true
!pip install -q huggingface_hub hf_transfer pandas numpy tqdm psutil
!wget -q https://antidote.cloud/f/ae5312aa983845c7abf1/?dl=1 -O llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q /content/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
%env HF_HUB_ENABLE_HF_TRANSFER=1
!hf download bartowski/Mistral-Nemo-Instruct-2407-GGUF Mistral-Nemo-Instruct-2407-Q5_K_M.gguf --local-dir .
print("Dependencies and model download completed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
env: HF_HUB_ENABLE_HF_TRANSFER=1
A new version of huggingface_hub (1.8.0) is available! You are using version 1.7.1.
To update, run: pip install -U huggingface_hub

Mistral-Nemo-Instruct-2407-Q5_K_M.gguf: 100% 8.73G/8.73G [01:01<00:00, 142MB/s]
Mistral-Nemo-Instruct-2407-Q5_K_M.gguf
Dependencies and model download completed.


In [2]:
# ============================================================
# 2. IMPORTS
# ============================================================
import os
import re
import sys
import json
import math
import time
import hashlib
import platform
import datetime
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 500)
pd.set_option("display.width", 200)


In [3]:
# ============================================================
# 3. GLOBAL CONFIG
# ============================================================
RANDOM_SEED = 1234
np.random.seed(RANDOM_SEED)

COMMUNITY_CARDS_FILE = "community_cards.json"
COMMUNITY_SUMMARY_FILE = "community_summary.csv"

MODEL_PATH = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
N_CTX = 2048
N_GPU_LAYERS = 35
N_THREADS = 8
N_BATCH = 256  # reduced for stability with grammar

USE_GBNF_GRAMMAR = True   # mandatory for your course requirement

MIN_COMMUNITY_SIZE_FOR_LLM = 15
MAX_COMMUNITIES_TO_PROCESS = None  # set integer only for debugging

CALLS_PER_COMMUNITY = 3
MAX_TOKENS_PER_CALL = 180   # reduced for grammar stability
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_RETRIES_PER_CALL = 2

BATCH_SIZE = 25
RUN_MODE = "selected_batches"   # "all_remaining" or "selected_batches"
START_BATCH_ID = 18
BATCH_IDS = [18,19,20,21]              # used only if RUN_MODE == "selected_batches"
SAVE_EVERY = 1

OUTPUT_DIR = Path("notebook3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Notebook 3 config loaded.")


Notebook 3 config loaded.


In [4]:
# ============================================================
# 4. LOGS FOR REPRODUCIBILITY
# ============================================================
seed_info = {"random_seed": RANDOM_SEED}
with open(OUTPUT_DIR / "seed_log_notebook3.json", "w", encoding="utf-8") as f:
    json.dump(seed_info, f, indent=2)

env_log = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "working_directory": os.getcwd(),
    "packages": {}
}

for pkg in ["pandas", "numpy", "tqdm", "psutil", "hashlib", "llama_cpp"]:
    try:
        m = __import__(pkg)
        env_log["packages"][pkg] = getattr(m, "__version__", "installed")
    except Exception:
        env_log["packages"][pkg] = "NOT INSTALLED"

with open(OUTPUT_DIR / "environment_log_notebook3.json", "w", encoding="utf-8") as f:
    json.dump(env_log, f, indent=2, ensure_ascii=False)

print("Saved notebook 3 seed/environment logs.")


Saved notebook 3 seed/environment logs.


In [5]:
# ============================================================
# 5. INPUT CHECKS + HASHES
# ============================================================
assert os.path.exists(COMMUNITY_CARDS_FILE), f"Missing file: {COMMUNITY_CARDS_FILE}"
assert os.path.exists(COMMUNITY_SUMMARY_FILE), f"Missing file: {COMMUNITY_SUMMARY_FILE}"

def sha256_of_file(filepath, chunk_size=8192):
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha256.update(chunk)
    return sha256.hexdigest()

input_hashes = {
    COMMUNITY_CARDS_FILE: sha256_of_file(COMMUNITY_CARDS_FILE),
    COMMUNITY_SUMMARY_FILE: sha256_of_file(COMMUNITY_SUMMARY_FILE)
}

with open(OUTPUT_DIR / "input_hashes_notebook3.json", "w", encoding="utf-8") as f:
    json.dump(input_hashes, f, indent=2)

print(json.dumps(input_hashes, indent=2))


{
  "community_cards.json": "05c7f2440b00140a4f11e0790f8027c8c994b679b292e6c226af6d64856176ff",
  "community_summary.csv": "3b394d75bb2d1c1c0ceb1861956550813b302d6687f57c3d598b93c0de01903f"
}


In [6]:
# ============================================================
# 6. LOAD NOTEBOOK 2 OUTPUTS
# ============================================================
with open(COMMUNITY_CARDS_FILE, "r", encoding="utf-8") as f:
    community_cards = json.load(f)

community_summary_df = pd.read_csv(COMMUNITY_SUMMARY_FILE, engine="python", on_bad_lines="skip")

print(f"Loaded community cards: {len(community_cards):,}")
print(f"Loaded community summary shape: {community_summary_df.shape}")
display(community_summary_df.head(3))


Loaded community cards: 1,307
Loaded community summary shape: (1307, 42)


,community_id,community_size,dominant_experience_reference,reference_share_junior,reference_share_mid,reference_share_senior,top_title_keywords,rep_job_id_1,rep_text_1,rep_similarity_1,rep_job_id_2,rep_text_2,rep_similarity_2,rep_job_id_3,rep_text_3,rep_similarity_3,rep_job_id_4,rep_text_4,rep_similarity_4,rep_job_id_5,rep_text_5,rep_similarity_5,avg_min_years_extracted,avg_max_years_extracted,avg_has_years_mention,avg_junior_term_count,avg_mid_term_count,avg_senior_term_count,avg_has_leadership_terms,avg_has_ownership_terms,avg_has_strategy_terms,avg_has_support_terms,avg_requires_bachelor,avg_requires_master,avg_requires_phd,avg_requires_certification,avg_mentions_fast_paced,avg_mentions_deadlines,avg_mentions_independent_work,avg_mentions_client_management,avg_entry_barrier_score,avg_semantic_text_word_count
0,0,1858,mid,0.298,0.686,0.017,"nurse, registered, care, licensed, lpn, practical, nursing, per, days, assistant, unit, med",3884831783,rn registered nurse agency free facility [SEP] position details employment status full-timeshift day shift 1000 sign on bonus recent pay scale increase 35- 37.50 depending on experience. come and grow with us we are affiliated with life care centers of america which is a privately owned 47-year industry leader in healthcare with more than 200 locations across the u.s. we are currently seeking a qualified registered nurse to add to our team of professionals. we believe that every job in our c...,0.9132,3884832746,rn registered nurse [SEP] position details employment status full-timeshift night shift come meet our new leadership team come and grow with us we are affiliated with life care centers of america which is a privately owned 47-year industry leader in healthcare with more than 200 locations across the u.s. we are currently seeking a qualified registered nurse to add to our team of professionals. we believe that every job in our company plays a vital role in our continued growth and commitment ...,0.9048,3884835418,rn registered nurse 4 000 sign-on bonus [SEP] position details employment status full-timesalary 32.00 hourlyshift 11pm-7amstatus full-time live the mission come and grow with us we are affiliated with life care centers of america which is a privately owned 47-year industry leader in healthcare with more than 200 locations across the u.s. we are currently seeking a qualified registered nurse to add to our team of professionals. we believe that every job in our company plays a vital role in o...,0.9016,3884833593,rn registered nurse 4 000 sign-on bonus [SEP] position details employment status full-timesalary 32.00 hourlyshift 3pm-11pmstatus full-time live the mission come and grow with us we are affiliated with life care centers of america which is a privately owned 47-year industry leader in healthcare with more than 200 locations across the u.s. we are currently seeking a qualified registered nurse to add to our team of professionals. we believe that every job in our company plays a vital role in o...,0.9007,3884832790,rn registered nurse 4 000 sign-on bonus [SEP] position details employment status full-timesalary 32.00 hourlyshift 7am-3pmstatus full-time live the mission come and grow with us we are affiliated with life care centers of america which is a privately owned 47-year industry leader in healthcare with more than 200 locations across the u.s. we are currently seeking a qualified registered nurse to add to our team of professionals. we believe that every job in our company plays a vital role in ou...,0.8994,2.912,3.326,0.165,1.466,0.895,0.679,0.303,0.321,0.379,0.679,0.164,0.043,0.008,0.882,0.008,0.020,0.114,0.368,2.170,504.443
1,1,1308,junior,0.492,0.440,0.068,"sales, manager, representative, salesperson, marketing, development, retail, business, outside, customer, specialist, director",3902790352,sales representative - packaging [SEP] job purpose our sales representative will be responsible for developing and executing the sales strategy for selling veritiv pr

In [7]:
# ============================================================
# 7. FILTER COMMUNITIES FOR LLM STAGE
# ============================================================
community_cards = [c for c in community_cards if int(c["community_size"]) >= MIN_COMMUNITY_SIZE_FOR_LLM]

if MAX_COMMUNITIES_TO_PROCESS is not None:
    community_cards = community_cards[:MAX_COMMUNITIES_TO_PROCESS]

community_cards = sorted(community_cards, key=lambda x: (-int(x["community_size"]), int(x["community_id"])))

valid_ids = {int(c["community_id"]) for c in community_cards}
community_summary_df = community_summary_df[community_summary_df["community_id"].astype(int).isin(valid_ids)].copy()

print(f"Communities kept for Notebook 3: {len(community_cards):,}")
print(f"Total represented rows in these communities: {sum(int(c['community_size']) for c in community_cards):,}")


Communities kept for Notebook 3: 1,307
Total represented rows in these communities: 55,545


In [8]:
# ============================================================
# 8. LOAD LOCAL LLM + GBNF GRAMMAR
# ============================================================
from llama_cpp import Llama, LlamaGrammar

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    n_threads=N_THREADS,
    n_batch=N_BATCH,
    use_mmap=True,
    use_mlock=False,
    verbose=False,
    seed=RANDOM_SEED
)

LABELING_SCHEMA = {
    "type": "object",
    "properties": {
        "community_id": {"type": "integer"},
        "dominant_experience_level": {"type": "string", "enum": ["junior", "mid", "senior"]},
        "entry_accessibility": {"type": "string", "enum": ["clear_entry_point", "moderate_entry_barrier", "restricted_entry_point"]},
        "community_label": {"type": "string"},
        "defining_skills": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 3,
            "maxItems": 3
        },
        "defining_responsibility_signals": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 2,
            "maxItems": 2
        },
        "confidence": {"type": "string", "enum": ["low", "medium", "high"]}
    },
    "required": [
        "community_id",
        "dominant_experience_level",
        "entry_accessibility",
        "community_label",
        "defining_skills",
        "defining_responsibility_signals",
        "confidence"
    ]
}

grammar = LlamaGrammar.from_json_schema(json.dumps(LABELING_SCHEMA))

print("LLM loaded successfully.")
print("Grammar loaded successfully.")


llama_context: n_ctx_per_seq (2048) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


LLM loaded successfully.
Grammar loaded successfully.


## Prompt design
This version keeps the prompt short on purpose.

Why:
- shorter prompt = lower crash risk with grammar
- shorter output = more stable JSON-constrained generation
- still enough community context to follow the professor’s method


In [9]:
# ============================================================
# 9. PROMPT TEMPLATE (SHORTER FOR GRAMMAR STABILITY)
# ============================================================
REP_CHAR_LIMIT = 350

PROMPT_TEMPLATE = '''[INST]
Analyze this semantic job community and return ONLY valid JSON.

Rules:
- infer from language, requirements, and responsibilities
- do not treat platform labels as ground truth
- keep label short
- keep skills and responsibilities short
- return only JSON

community_id: {community_id}
community_size: {community_size}
top_keywords: {top_title_keywords}
junior_share: {junior_share}
mid_share: {mid_share}
senior_share: {senior_share}

signals:
{signals_text}

representative_rank: {rep_rank}
representative_text:
{rep_text}
[/INST]'''

def build_prompt(card, rep_text, rep_rank):
    distribution = card.get("reference_distribution", {})
    signals = card.get("signals", {})

    signal_keys = [
        "avg_min_years_extracted",
        "avg_max_years_extracted",
        "avg_has_leadership_terms",
        "avg_requires_bachelor",
        "avg_requires_master",
        "avg_requires_certification",
        "avg_entry_barrier_score"
    ]

    signal_lines = []
    for k in signal_keys:
        if k in signals:
            signal_lines.append(f"- {k}: {signals[k]}")
    signals_text = "\n".join(signal_lines)

    return PROMPT_TEMPLATE.format(
        community_id=int(card["community_id"]),
        community_size=int(card["community_size"]),
        top_title_keywords=card.get("top_title_keywords", ""),
        junior_share=distribution.get("junior_share", 0.0),
        mid_share=distribution.get("mid_share", 0.0),
        senior_share=distribution.get("senior_share", 0.0),
        signals_text=signals_text,
        rep_rank=rep_rank,
        rep_text=str(rep_text)[:REP_CHAR_LIMIT]
    )


In [10]:
# ============================================================
# 10. PARSING + VALIDATION
# ============================================================
VALID_LEVELS = {"junior", "mid", "senior"}
VALID_ACCESS = {"clear_entry_point", "moderate_entry_barrier", "restricted_entry_point"}
VALID_CONFIDENCE = {"low", "medium", "high"}

def normalize_list(value, target_len):
    if not isinstance(value, list):
        value = []
    value = [str(x).strip() for x in value if str(x).strip()]
    value = value[:target_len]
    while len(value) < target_len:
        value.append("unknown")
    return value

def validate_output(obj, expected_community_id):
    if not isinstance(obj, dict):
        return None

    try:
        community_id = int(obj.get("community_id", expected_community_id))
    except Exception:
        community_id = expected_community_id

    level = str(obj.get("dominant_experience_level", "")).strip().lower()
    access = str(obj.get("entry_accessibility", "")).strip().lower()
    label = str(obj.get("community_label", "")).strip()
    skills = normalize_list(obj.get("defining_skills", []), 3)
    responsibilities = normalize_list(obj.get("defining_responsibility_signals", []), 2)
    confidence = str(obj.get("confidence", "")).strip().lower()

    if level not in VALID_LEVELS:
        return None
    if access not in VALID_ACCESS:
        return None
    if confidence not in VALID_CONFIDENCE:
        return None
    if not label:
        return None

    return {
        "community_id": community_id,
        "dominant_experience_level": level,
        "entry_accessibility": access,
        "community_label": label[:100],
        "defining_skills": skills,
        "defining_responsibility_signals": responsibilities,
        "confidence": confidence
    }


In [11]:
# ============================================================
# 11. GRAMMAR TEST (MANDATORY BEFORE FULL RUN)
# ============================================================
test_card = community_cards[0]
test_rep = test_card["representatives"][0]
test_prompt = build_prompt(test_card, test_rep, rep_rank=1)

test_response = llm(
    test_prompt,
    max_tokens=MAX_TOKENS_PER_CALL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    echo=False,
    grammar=grammar
)

test_raw = test_response["choices"][0]["text"]
print("RAW OUTPUT:")
print(test_raw)

test_obj = json.loads(test_raw)
test_parsed = validate_output(test_obj, int(test_card["community_id"]))

print("\nPARSED OUTPUT:")
print(test_parsed)

assert test_parsed is not None, "Grammar test failed."
print("Grammar test passed.")


RAW OUTPUT:
{"community_id": 0, "dominant_experience_level": "mid", "entry_accessibility": "moderate_entry_barrier", "community_label": "Nursing", "defining_skills": ["nursing", "medical care", "patient care"], "defining_responsibility_signals": ["leadership", "certification"] , "confidence": "high"}

PARSED OUTPUT:
{'community_id': 0, 'dominant_experience_level': 'mid', 'entry_accessibility': 'moderate_entry_barrier', 'community_label': 'Nursing', 'defining_skills': ['nursing', 'medical care', 'patient care'], 'defining_responsibility_signals': ['leadership', 'certification'], 'confidence': 'high'}
Grammar test passed.


In [12]:
# ============================================================
# 12. INFERENCE HELPERS
# ============================================================
CONFIDENCE_NUM = {"low": 1, "medium": 2, "high": 3}

def run_single_call(card, rep_text, rep_rank):
    community_id = int(card["community_id"])
    prompt = build_prompt(card, rep_text, rep_rank)

    last_raw = ""
    for attempt in range(1, MAX_RETRIES_PER_CALL + 1):
        try:
            response = llm(
                prompt,
                max_tokens=MAX_TOKENS_PER_CALL,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                echo=False,
                grammar=grammar
            )
            raw_text = response["choices"][0]["text"]
            last_raw = raw_text
            parsed_obj = json.loads(raw_text)
            parsed = validate_output(parsed_obj, community_id)
            if parsed is not None:
                parsed["rep_rank"] = rep_rank
                parsed["attempt"] = attempt
                parsed["raw_output"] = raw_text
                return parsed
        except Exception as e:
            last_raw = f"ERROR: {str(e)}"

    return {
        "community_id": community_id,
        "dominant_experience_level": "mid",
        "entry_accessibility": "moderate_entry_barrier",
        "community_label": "manual_review_required",
        "defining_skills": ["unknown", "unknown", "unknown"],
        "defining_responsibility_signals": ["unknown", "unknown"],
        "confidence": "low",
        "rep_rank": rep_rank,
        "attempt": MAX_RETRIES_PER_CALL,
        "raw_output": last_raw
    }

def majority_vote(values, default_value):
    clean = [v for v in values if v is not None]
    if not clean:
        return default_value, 0.0
    counts = Counter(clean)
    label, count = counts.most_common(1)[0]
    return label, round(count / len(clean), 3)

def aggregate_call_results(card, call_results):
    levels = [r["dominant_experience_level"] for r in call_results]
    accesses = [r["entry_accessibility"] for r in call_results]
    labels = [r["community_label"] for r in call_results]
    confidences = [r["confidence"] for r in call_results]

    final_level, level_agreement = majority_vote(levels, "mid")
    final_access, access_agreement = majority_vote(accesses, "moderate_entry_barrier")
    final_label, label_agreement = majority_vote(labels, "manual_review_required")

    skill_counter = Counter()
    resp_counter = Counter()
    for r in call_results:
        skill_counter.update([s for s in r["defining_skills"] if s != "unknown"])
        resp_counter.update([s for s in r["defining_responsibility_signals"] if s != "unknown"])

    final_skills = [s for s, _ in skill_counter.most_common(3)]
    final_resp = [s for s, _ in resp_counter.most_common(2)]

    while len(final_skills) < 3:
        final_skills.append("unknown")
    while len(final_resp) < 2:
        final_resp.append("unknown")

    avg_conf = np.mean([CONFIDENCE_NUM.get(c, 1) for c in confidences]) if confidences else 1.0
    if avg_conf >= 2.5:
        final_conf = "high"
    elif avg_conf >= 1.75:
        final_conf = "medium"
    else:
        final_conf = "low"

    overall_agreement = round(np.mean([level_agreement, access_agreement, label_agreement]), 3)

    return {
        "community_id": int(card["community_id"]),
        "community_size": int(card["community_size"]),
        "dominant_experience_level": final_level,
        "entry_accessibility": final_access,
        "community_label": final_label,
        "defining_skills": final_skills,
        "defining_responsibility_signals": final_resp,
        "confidence": final_conf,
        "level_agreement": level_agreement,
        "access_agreement": access_agreement,
        "label_agreement": label_agreement,
        "overall_agreement": overall_agreement,
        "calls_used": len(call_results),
        "top_title_keywords": card.get("top_title_keywords", "")
    }

def label_one_community(card):
    reps = [r for r in card.get("representatives", []) if isinstance(r, str) and r.strip()]
    if not reps:
        reps = ["No representative text available."]
    reps = reps[:CALLS_PER_COMMUNITY]

    call_results = []
    for rep_rank, rep_text in enumerate(reps, start=1):
        call_results.append(run_single_call(card, rep_text, rep_rank))

    aggregated = aggregate_call_results(card, call_results)
    aggregated["raw_call_outputs"] = json.dumps(call_results, ensure_ascii=False)
    return aggregated


In [13]:
# ============================================================
# 13. BATCH CONFIGURATION
# ============================================================
total_batches = math.ceil(len(community_cards) / BATCH_SIZE)

if RUN_MODE == "all_remaining":
    batches_to_run = list(range(START_BATCH_ID, total_batches + 1))
else:
    batches_to_run = sorted(set([b for b in BATCH_IDS if 1 <= b <= total_batches]))

print(f"Total communities to process: {len(community_cards):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Total batches: {total_batches}")
print(f"Run mode: {RUN_MODE}")
print(f"Batches to run now: {batches_to_run}")


Total communities to process: 1,307
Batch size: 25
Total batches: 53
Run mode: selected_batches
Batches to run now: [18, 19, 20, 21]


In [14]:
# ============================================================
# 14. PER-BATCH FILE HELPERS
# ============================================================
def batch_paths(batch_id):
    return {
        "checkpoint": OUTPUT_DIR / f"checkpoint_batch_{batch_id:02d}.json",
        "results_json": OUTPUT_DIR / f"results_batch_{batch_id:02d}.json",
        "results_csv": OUTPUT_DIR / f"results_batch_{batch_id:02d}.csv",
        "meta_json": OUTPUT_DIR / f"batch_meta_{batch_id:02d}.json"
    }


In [15]:
# ============================================================
# 15. RUN A SINGLE BATCH (RESUMABLE)
# ============================================================
def run_single_batch(batch_id):
    start_idx = (batch_id - 1) * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(community_cards))
    batch_cards = community_cards[start_idx:end_idx]
    paths = batch_paths(batch_id)

    print("=" * 70)
    print(f"RUNNING BATCH {batch_id}/{total_batches}")
    print(f"Communities in batch: {len(batch_cards)}")
    print("=" * 70)

    start_row = 0
    results = []

    if os.path.exists(paths["results_json"]):
        with open(paths["results_json"], "r", encoding="utf-8") as f:
            results = json.load(f)
        print(f"Loaded existing results for batch {batch_id}: {len(results)}")

    if os.path.exists(paths["checkpoint"]):
        with open(paths["checkpoint"], "r", encoding="utf-8") as f:
            ckpt = json.load(f)
        start_row = int(ckpt.get("last_processed_row", 0))

    if start_row != len(results):
        start_row = len(results)

    batch_start_time = time.time()

    for i in tqdm(range(start_row, len(batch_cards)), desc=f"Batch {batch_id}"):
        card = batch_cards[i]
        labeled = label_one_community(card)
        labeled["batch_id"] = batch_id
        results.append(labeled)

        if ((i + 1) % SAVE_EVERY == 0) or ((i + 1) == len(batch_cards)):
            with open(paths["checkpoint"], "w", encoding="utf-8") as f:
                json.dump({"last_processed_row": i + 1}, f, indent=2)
            with open(paths["results_json"], "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            pd.DataFrame(results).to_csv(paths["results_csv"], index=False)

    elapsed = time.time() - batch_start_time

    meta = {
        "batch_id": batch_id,
        "community_count": len(batch_cards),
        "processed_results": len(results),
        "elapsed_seconds": round(elapsed, 2),
        "avg_seconds_per_community": round(elapsed / max(len(batch_cards), 1), 2)
    }

    with open(paths["meta_json"], "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print(json.dumps(meta, indent=2))
    return meta


In [16]:
# ============================================================
# 16. RUN REQUESTED BATCHES
# ============================================================
all_meta = []
for batch_id in batches_to_run:
    all_meta.append(run_single_batch(batch_id))

with open(OUTPUT_DIR / "run_session_meta.json", "w", encoding="utf-8") as f:
    json.dump(all_meta, f, indent=2)

print("All requested batches completed.")


RUNNING BATCH 18/53
Communities in batch: 25


Batch 18: 100%|██████████| 25/25 [36:41<00:00, 88.05s/it]


{
  "batch_id": 18,
  "community_count": 25,
  "processed_results": 25,
  "elapsed_seconds": 2201.19,
  "avg_seconds_per_community": 88.05
}
RUNNING BATCH 19/53
Communities in batch: 25


Batch 19: 100%|██████████| 25/25 [36:42<00:00, 88.10s/it]


{
  "batch_id": 19,
  "community_count": 25,
  "processed_results": 25,
  "elapsed_seconds": 2202.45,
  "avg_seconds_per_community": 88.1
}
RUNNING BATCH 20/53
Communities in batch: 25


Batch 20: 100%|██████████| 25/25 [37:00<00:00, 88.81s/it]


{
  "batch_id": 20,
  "community_count": 25,
  "processed_results": 25,
  "elapsed_seconds": 2220.23,
  "avg_seconds_per_community": 88.81
}
RUNNING BATCH 21/53
Communities in batch: 25


Batch 21: 100%|██████████| 25/25 [36:10<00:00, 86.81s/it]

{
  "batch_id": 21,
  "community_count": 25,
  "processed_results": 25,
  "elapsed_seconds": 2170.29,
  "avg_seconds_per_community": 86.81
}
All requested batches completed.


In [17]:
# ============================================================
# 17. MERGE ALL COMPLETED BATCHES
# ============================================================
batch_csvs = sorted(OUTPUT_DIR.glob("results_batch_*.csv"))
print(f"Found completed batch CSVs: {len(batch_csvs)}")

if not batch_csvs:
    raise ValueError("No completed batch result CSVs found.")

merged_labels_df = pd.concat([pd.read_csv(f) for f in batch_csvs], ignore_index=True)
merged_labels_df = merged_labels_df.sort_values("community_id").drop_duplicates(subset=["community_id"], keep="last").reset_index(drop=True)

merged_labels_df.to_csv(OUTPUT_DIR / "all_community_labels_merged.csv", index=False)
print("Saved: all_community_labels_merged.csv")
display(merged_labels_df.head(5))


Found completed batch CSVs: 4
Saved: all_community_labels_merged.csv


,community_id,community_size,dominant_experience_level,entry_accessibility,community_label,defining_skills,defining_responsibility_signals,confidence,level_agreement,access_agreement,label_agreement,overall_agreement,calls_used,top_title_keywords,raw_call_outputs,batch_id
0,425,30,mid,moderate_entry_barrier,Finance & Operations,"['Financial Analysis', 'Operations Management', 'Cost Analysis']","['Leadership', 'Strategic Planning']",high,0.667,1.0,0.667,0.778,3,"financial, analyst, finance, manager, senior, chief, officer, plant, systems, cost, vice, president","[{""community_id"": 425, ""dominant_experience_level"": ""mid"", ""entry_accessibility"": ""moderate_entry_barrier"", ""community_label"": ""Finance & Operations"", ""defining_skills"": [""Financial Analysis"", ""Operations Management"", ""Cost Analysis""], ""defining_responsibility_signals"": [""Leadership"", ""Strategic Planning""], ""confidence"": ""high"", ""rep_rank"": 1, ""attempt"": 1, ""raw_output"": ""{\""community_id\"": 425, \""dominant_experience_level\"": \""mid\"", \""entry_accessibility\"": \""moderate_entry_barrier\"", \""co...",18
1,426,30,mid,moderate_entry_barrier,Executive Administrative Assistants,"['collaboration', 'outlook', 'project_management']","['collaboration', 'technical_support']",high,1.000,1.0,0.333,0.778,3,"assistant, administrative, executive, admin, administrator, senior, support, sci, office, somerset, collaboration, technical","[{""community_id"": 426, ""dominant_experience_level"": ""mid"", ""entry_accessibility"": ""moderate_entry_barrier"", ""community_label"": ""Executive Administrative Assistants"", ""defining_skills"": [""outlook"", ""project_management"", ""event_management""], ""defining_responsibility_signals"": [""collaboration"", ""technical_support""], ""confidence"": ""high"", ""rep_rank"": 1, ""attempt"": 1, ""raw_output"": ""{\""community_id\"": 426, \""dominant_experience_level\"": \""mid\"", \""entry_accessibility\"": \""moderate_entry_barrier\""...",18
2,427,30,mid,moderate_entry_barrier,Customer Service,"['customer service', 'client support', 'dedicated assistance']","['customer greeting', 'service provision']",high,1.000,1.0,1.000,1.000,3,"customer, representative, service, manager, assistant, credit, specialist, support, client, services, dedicated, transportation","[{""community_id"": 427, ""dominant_experience_level"": ""mid"", ""entry_accessibility"": ""moderate_entry_barrier"", ""community_label"": ""Customer Service"", ""defining_skills"": [""customer service"", ""client support"", ""service management""], ""defining_responsibility_signals"": [""customer greeting"", ""service provision""], ""confidence"": ""high"", ""rep_rank"": 1, ""attempt"": 1, ""raw_output"": ""{\""community_id\"": 427, \""dominant_experience_level\"": \""mid\"", \""entry_accessibility\"": \""moderate_entry_barrier\"", \""comm...",18
3,428,30,junior,moderate_entry_barrier,Hotel Operations Management,"['sales', 'leadership', 'customer_service']","['strategic_financial_management', 'departmental_supervision']",high,1.000,1.0,0.667,0.889,3,"sales, director, attendant, rooms, guest, manager, specialist, supervisor, coordinator, oem, assistant, hotel","[{""community_id"": 428, ""dominant_experience_level"": ""junior"", ""entry_accessibility"": ""moderate_entry_barrier"", ""community_label"": ""Hotel Operations"", ""defining_skills"": [""sales"", ""guest management"", ""room management""], ""defining_responsibility_signals"": [""day-to-day operations"", ""activity management""], ""confidence"": ""high"", ""rep_rank"": 1, ""attempt"": 1, ""raw_output"": ""{\""community_id\"": 428, \""dominant_experience_level\"": \""junior\"", \""entry_accessibility\"": \""moderate_entry_barrier\"", \""comm...",18
4,429,30,junior,moderate_entry_barrier,Controls Engineer,"['automation', 'sheet metal fabrication', 'process improvement']","['lead development', 'implement manufacturing process']",high,1.000,1.0,0.333,0.778,3,"engineer, project, controls, manager, e

## Stage 3 intelligence generation
This is where the notebook turns labels into **operational outputs**.


In [18]:
# ============================================================
# 18. JOIN NOTEBOOK 2 SUMMARY WITH NOTEBOOK 3 LABELS
# ============================================================
stage3_df = community_summary_df.merge(
    merged_labels_df,
    on="community_id",
    how="inner",
    suffixes=("_nb2", "_nb3")
)

print("Merged stage3 dataframe shape:", stage3_df.shape)
display(stage3_df.head(3))


Merged stage3 dataframe shape: (100, 57)


,community_id,community_size_nb2,dominant_experience_reference,reference_share_junior,reference_share_mid,reference_share_senior,top_title_keywords_nb2,rep_job_id_1,rep_text_1,rep_similarity_1,rep_job_id_2,rep_text_2,rep_similarity_2,rep_job_id_3,rep_text_3,rep_similarity_3,rep_job_id_4,rep_text_4,rep_similarity_4,rep_job_id_5,rep_text_5,rep_similarity_5,avg_min_years_extracted,avg_max_years_extracted,avg_has_years_mention,avg_junior_term_count,avg_mid_term_count,avg_senior_term_count,avg_has_leadership_terms,avg_has_ownership_terms,avg_has_strategy_terms,avg_has_support_terms,avg_requires_bachelor,avg_requires_master,avg_requires_phd,avg_requires_certification,avg_mentions_fast_paced,avg_mentions_deadlines,avg_mentions_independent_work,avg_mentions_client_management,avg_entry_barrier_score,avg_semantic_text_word_count,community_size_nb3,dominant_experience_level,entry_accessibility,community_label,defining_skills,defining_responsibility_signals,confidence,level_agreement,access_agreement,label_agreement,overall_agreement,calls_used,top_title_keywords_nb3,raw_call_outputs,batch_id
0,431,30,junior,0.567,0.433,0.000,"technician, quality, inspector, shift, control, ndt, bayway, refinery, 2nd, source, assurance, manufacturing",3902740628,inspector bayway refinery [SEP] phillips 66 you - together we can fuel the future the inspector is responsible for maintaining the mechanical integrity of fixed equipment and piping in the refinery by performing required inspections at timely intervals. inspectors examine inspect collect information and evaluate inspection data. they also provide recommendations under the guidance of an inspection supervisor or chief inspector. responsibilities may include performing and coordinating require...,0.8028,3903803260,inspector bayway refinery [SEP] phillips 66 you - together we can fuel the future the inspector is responsible for maintaining the mechanical integrity of fixed equipment and piping in the refinery by performing required inspections at timely intervals. inspectors examine inspect collect information and evaluate inspection data. they also provide recommendations under the guidance of an inspection supervisor or chief inspector. responsibilities may include performing and coordinating require...,0.8028,3891070815,quality inspector [SEP] job title quality inspector iii cmm location multiple locations huntsville al kent wa merritt island flshift multiple shiftstype contractcompensation range 42 - 49/hr - depends on shift and locationabout the role we are seeking several skilled quality inspectors level iii to join our dynamic team. this role focuses on the precision inspection of aerospace components using coordinate measuring machines cmm . candidates should have a strong foundation in geometric dimen...,0.8026,3904419294,rt level ii technician [SEP] acuren is currently recruiting for level ii rt technician for our angleton tx operations. responsibilities set up and calibrate nondestructive testing equipment conduct tests to ensure quality or detect discontinuities defects using ndt methods of inspection establish techniques for proper examination of objects under inspection ensuring strict adherence to safety regulations apply testing criteria in accordance with applicable specifications or standards and eva...,0.7973,3902851177,quality inspector 2nd shift [SEP] we are currently interviewing for a quality inspector 2 nd shift in berlin ct bodycote is offering a 1500 sign-on retention bonus bodycote offers paid holidays and paid time off401k with a matchmedical dental and vision plans for employee and familymust pass a pre-employment drug screenability to work full-time 40 hours per week3 00 pm- 11 30 pm monday - friday our people are the heart of our business. as the world s largest provider of thermal processing se...,0.7933,5.545,5.909,0.367,1.367,1.6,0.467,0.200,0.333,0.567,0.533,0.100,0.067,0.000,0.533,0.033,0.100,0.167,0.700,2.433,472.033,30,junior,moderate_entry_barrier,Refinery Inspe

In [20]:
# ============================================================
# 19. DETERMINISTIC ACTIONABLE INTELLIGENCE RULES
# ============================================================
def build_analyst_interpretation(row):
    level = row["dominant_experience_level"]
    access = row["entry_accessibility"]
    label = row["community_label"]
    agreement = float(row["overall_agreement"])

    if access == "clear_entry_point" and level == "junior":
        return f"{label} is a high-value entry-point community with relatively accessible labor-market conditions."
    if access == "restricted_entry_point" and level == "junior":
        return f"{label} is junior-framed but behaves like a restricted-entry segment that may discourage genuine entry candidates."
    if access == "restricted_entry_point":
        return f"{label} is a restricted-entry community with elevated barriers and should be monitored as a high-selectivity segment."
    if agreement < 0.67:
        return f"{label} is semantically ambiguous and should be reviewed manually before relying on it."
    return f"{label} is a moderate-barrier community representing a stable labor-market segment."

def build_recruiter_action(row):
    access = row["entry_accessibility"]
    level = row["dominant_experience_level"]
    skills = ", ".join(row["defining_skills"])

    if access == "clear_entry_point" and level == "junior":
        return f"Prioritize this community for entry-level sourcing and screen for {skills}."
    if access == "restricted_entry_point" and level == "junior":
        return "Audit whether the role mixes junior framing with advanced requirements."
    if access == "restricted_entry_point":
        return "Use this community as a benchmark for selective hiring and communicate barriers clearly."
    return f"Use this community to refine hiring criteria around {skills}."

def build_candidate_advice(row):
    access = row["entry_accessibility"]
    skills = ", ".join(row["defining_skills"])
    responsibilities = ", ".join(row["defining_responsibility_signals"])

    if access == "clear_entry_point":
        return f"This is a realistic entry route. Show evidence of {skills} and {responsibilities}."
    if access == "restricted_entry_point":
        return f"This is not an easy entry route. Strengthen proof of {skills} before targeting it."
    return f"This segment is moderately accessible. Improve evidence of {skills} and {responsibilities}."

def build_policy_signal(row):
    access = row["entry_accessibility"]
    size = int(row["community_size"])
    if access == "clear_entry_point" and size >= 100:
        return "Potential workforce-entry channel worth monitoring for access and transition programs."
    if access == "restricted_entry_point" and size >= 100:
        return "Potential barrier-heavy segment worth monitoring for qualification inflation."
    return "Monitor as a secondary labor-market segment."

def build_review_flag(row):
    if float(row["overall_agreement"]) < 0.67:
        return "manual_review_recommended"
    if row["confidence"] == "low":
        return "manual_review_recommended"
    return "ok"

stage3_df["analyst_interpretation"] = stage3_df.apply(build_analyst_interpretation, axis=1)
stage3_df["recruiter_action"] = stage3_df.apply(build_recruiter_action, axis=1)
stage3_df["candidate_advice"] = stage3_df.apply(build_candidate_advice, axis=1)
stage3_df["policy_signal"] = stage3_df.apply(build_policy_signal, axis=1)
stage3_df["review_flag"] = stage3_df.apply(build_review_flag, axis=1)

display(stage3_df[[
    "community_id", "community_size", "community_label",
    "dominant_experience_level", "entry_accessibility",
    "overall_agreement", "review_flag"
]].head(10))


KeyError: 'community_size'

In [21]:
# ============================================================
# 20. SAVE RANKED OUTPUTS
# ============================================================
entry_df = stage3_df[
    (stage3_df["entry_accessibility"] == "clear_entry_point") &
    (stage3_df["dominant_experience_level"] == "junior")
].copy().sort_values(by=["overall_agreement", "community_size"], ascending=[False, False]).reset_index(drop=True)

restricted_df = stage3_df[
    stage3_df["entry_accessibility"] == "restricted_entry_point"
].copy().sort_values(by=["overall_agreement", "community_size"], ascending=[False, False]).reset_index(drop=True)

ambiguous_df = stage3_df[
    stage3_df["review_flag"] == "manual_review_recommended"
].copy().sort_values(by=["overall_agreement", "community_size"], ascending=[True, False]).reset_index(drop=True)

entry_df.to_csv(OUTPUT_DIR / "entry_point_communities_ranked.csv", index=False)
restricted_df.to_csv(OUTPUT_DIR / "restricted_entry_communities_ranked.csv", index=False)
ambiguous_df.to_csv(OUTPUT_DIR / "ambiguous_communities_for_review.csv", index=False)
stage3_df.to_csv(OUTPUT_DIR / "stage3_community_intelligence.csv", index=False)

print("Saved Stage 3 output files.")


KeyError: 'community_size'

In [22]:
# ============================================================
# 21. NOTEBOOK 3 REPRODUCIBILITY DOCUMENT
# ============================================================
notebook3_doc = {
    "input_files": {
        "community_cards_file": COMMUNITY_CARDS_FILE,
        "community_summary_file": COMMUNITY_SUMMARY_FILE,
        "hashes": input_hashes
    },
    "random_seed": RANDOM_SEED,
    "llm": {
        "model_path": MODEL_PATH,
        "n_ctx": N_CTX,
        "n_gpu_layers": N_GPU_LAYERS,
        "n_threads": N_THREADS,
        "n_batch": N_BATCH,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "calls_per_community": CALLS_PER_COMMUNITY,
        "use_gbnf_grammar": USE_GBNF_GRAMMAR
    },
    "community_inclusion": {
        "min_community_size_for_llm": MIN_COMMUNITY_SIZE_FOR_LLM,
        "communities_processed": int(len(community_cards))
    },
    "batching": {
        "batch_size": BATCH_SIZE,
        "run_mode": RUN_MODE
    }
}

with open(OUTPUT_DIR / "reproducibility_documentation_notebook3.json", "w", encoding="utf-8") as f:
    json.dump(notebook3_doc, f, indent=2)

print("Saved: reproducibility_documentation_notebook3.json")
print(json.dumps(notebook3_doc, indent=2))


Saved: reproducibility_documentation_notebook3.json
{
  "input_files": {
    "community_cards_file": "community_cards.json",
    "community_summary_file": "community_summary.csv",
    "hashes": {
      "community_cards.json": "05c7f2440b00140a4f11e0790f8027c8c994b679b292e6c226af6d64856176ff",
      "community_summary.csv": "3b394d75bb2d1c1c0ceb1861956550813b302d6687f57c3d598b93c0de01903f"
    }
  },
  "random_seed": 1234,
  "llm": {
    "model_path": "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf",
    "n_ctx": 2048,
    "n_gpu_layers": 35,
    "n_threads": 8,
    "n_batch": 256,
    "temperature": 0.0,
    "top_p": 1.0,
    "calls_per_community": 3,
    "use_gbnf_grammar": true
  },
  "community_inclusion": {
    "min_community_size_for_llm": 15,
    "communities_processed": 1307
  },
  "batching": {
    "batch_size": 25,
    "run_mode": "selected_batches"
  }
}


In [23]:
# ============================================================
# 22. FINAL SUMMARY
# ============================================================
print("=" * 70)
print("NOTEBOOK 3 COMPLETE")
print("=" * 70)
print(f"Communities processed: {len(community_cards):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Calls per community: {CALLS_PER_COMMUNITY}")
print()
print("Generated files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)


NOTEBOOK 3 COMPLETE
Communities processed: 1,307
Batch size: 25
Calls per community: 3

Generated files:
- all_community_labels_merged.csv
- batch_meta_18.json
- batch_meta_19.json
- batch_meta_20.json
- batch_meta_21.json
- checkpoint_batch_18.json
- checkpoint_batch_19.json
- checkpoint_batch_20.json
- checkpoint_batch_21.json
- environment_log_notebook3.json
- input_hashes_notebook3.json
- reproducibility_documentation_notebook3.json
- results_batch_18.csv
- results_batch_18.json
- results_batch_19.csv
- results_batch_19.json
- results_batch_20.csv
- results_batch_20.json
- results_batch_21.csv
- results_batch_21.json
- run_session_meta.json
- seed_log_notebook3.json
